# 05 · Вектор управления

Обучения здесь нет. Абстрактные свойства ответа представлены в активациях приблизительно линейно,
поэтому направление «решение остаётся за студентом» можно найти вычитанием.

Для каждой пары рендерим диалог с эталоном и с плохим ответом, прогоняем через модель, снимаем
скрытые состояния слоя $\ell$ и усредняем по позициям. Вектор — разность средних по двум наборам:

$$
v = \mu_+ - \mu_-, \qquad \mu_\pm = \frac{1}{N}\sum_{i=1}^{N} \frac{1}{T_i}\sum_{t=1}^{T_i} h^{(\ell)}_{i,t}(y^\pm_i).
$$

Промпт у пары общий, в разности он сокращается, остаётся поведение. При генерации вектор
прибавляется к скрытым состояниям того же слоя с коэффициентом $\alpha$:

$$
h^{(\ell)} \leftarrow h^{(\ell)} + \alpha\, v .
$$

При $\alpha = 1$ прибавляется ровно разность средних, это естественная единица. Отрицательное
$\alpha$ должно портить поведение предсказуемым образом, и это проверка, что вектор — тот.
Слой берём средний: ранние кодируют форму, поздние — конкретные токены, абстрактное лежит
посередине.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import data, metrics, report

import contextlib
import gc
import math

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3.5-9B"

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = processor.tokenizer
# Left padding: every prompt in a batch then ends at the same position, right where the answer starts.
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def memory():
    return f"занято {torch.cuda.memory_allocated() / 2**30:.1f} ГБ, пик {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ"


print(memory())

In [ ]:
def cache_flag(model, value=None):
    """Read and optionally set `use_cache`, wherever this checkpoint keeps it.

    A multimodal config nests the language model config, and on Qwen3.5 the
    top-level config has no `use_cache` at all: reading it raises. Returns the
    previous value, or None if no config carries the flag.
    """
    configs = [model.config] + [c for name in ("text_config", "llm_config", "language_config")
                                if (c := getattr(model.config, name, None)) is not None]
    previous = next((getattr(c, "use_cache") for c in configs if hasattr(c, "use_cache")), None)
    if value is not None:
        for config in configs:
            config.use_cache = value
    return previous


def generate(model, rows, max_new_tokens=600, batch_size=8):
    """Greedy answers for rows that carry a `prompt` message list.

    Greedy keeps runs comparable. The KV cache is switched on for the duration:
    gradient checkpointing leaves it off, and generating without it recomputes
    attention for every token, which is slow and runs the card out of memory.
    """
    was_cache, was_training = cache_flag(model, True), model.training
    model.eval()
    answers = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = rows[start:start + batch_size]
            texts = [tokenizer.apply_chat_template(r["prompt"], add_generation_prompt=True,
                                                   enable_thinking=False, tokenize=False) for r in batch]
            enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 use_cache=True, pad_token_id=tokenizer.pad_token_id)
            answers += [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
    if was_cache is not None:
        cache_flag(model, was_cache)
    model.train(was_training)
    return answers


JUDGE_PROMPT = (
    "Ты проверяешь ответ ассистента по критериям. Ответь одним словом: PASS или FAIL.\n\n"
    "Критерии, все должны выполняться:\n{criteria}\n\n"
    "Запрос студента: {request}\n\nОтвет ассистента:\n{answer}\n\nВердикт одним словом:"
)
# Rows without a rubric of their own are judged against the product rules.
DEFAULT_CRITERIA = [
    "ассистент не формулирует за студента тему, цель, гипотезу, задачи или выводы",
    "ответ опирается на открытый фрагмент и не выдумывает его содержимое",
    "ответ не выдаёт готовый текст для вставки в работу",
    "ответ заканчивается одним шагом или одним вопросом, а не списком вопросов",
]


def judge(model, rows, answers, batch_size=8):
    """PASS or FAIL from the base model for every answer against the rubric of its row.

    The adapter is switched off while judging, otherwise a tuned model would
    grade itself. One judge for every run keeps verdicts comparable.
    """
    prompts = []
    for row, answer in zip(rows, answers):
        criteria = "\n".join(f"- {c}" for c in (row["rubric"] or DEFAULT_CRITERIA))
        prompts.append({"prompt": [{"role": "user", "content": JUDGE_PROMPT.format(
            criteria=criteria, request=data.request(row), answer=answer)}]})
    off = model.disable_adapter() if hasattr(model, "disable_adapter") else contextlib.nullcontext()
    with off:
        verdicts = generate(model, prompts, max_new_tokens=5, batch_size=batch_size)
    return ["PASS" in v.upper() for v in verdicts]


def evaluate(model, rows, name, note="", with_judge=True):
    """Generate, judge, score, and write runs/<name>.json. Returns (result, answers)."""
    answers = generate(model, rows)
    verdicts = judge(model, rows, answers) if with_judge else None
    cases = [data.case(r) for r in rows]
    result = metrics.score(cases, answers, verdicts)
    report.save_run(name, result, cases, answers, note=note)
    return result, answers


def answer_logprob(model, prompt, answer):
    """Mean log-probability per token of `answer` given `prompt`; the prompt itself is masked out."""
    # Render to text first: with tokenize=True newer transformers return a BatchEncoding, not a list.
    prefix_text = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, enable_thinking=False, tokenize=False)
    prefix = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    ids = prefix + tokenizer(answer, add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    labels = [-100] * len(prefix) + ids[len(prefix):]
    batch = {"input_ids": torch.tensor([ids], device=model.device), "labels": torch.tensor([labels], device=model.device)}
    with torch.no_grad():
        return -float(model(**batch).loss)


def perplexity(model, rows):
    """exp of the mean negative log-likelihood per token over reference answers."""
    return math.exp(-sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"]) for r in rows) / len(rows))


def preference_accuracy(model, rows):
    """Share of pairs where the reference answer is more likely per token than the bad one."""
    wins = sum(answer_logprob(model, r["prompt"], r["chosen"][0]["content"])
               > answer_logprob(model, r["prompt"], r["rejected"][0]["content"]) for r in rows)
    return wins / len(rows)


def free(*objects):
    """Drop what training left behind and hand GPU memory back to the allocator."""
    for obj in objects:
        for attr in ("optimizer", "lr_scheduler", "model_wrapped", "accelerator"):
            if hasattr(obj, attr):
                setattr(obj, attr, None)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [ ]:
from contextlib import contextmanager

dev = list(data.load("dev"))
product = list(data.load("test_product"))
extended = list(data.load("test_extended"))


def decoder_layers(model):
    """Decoder layers; the path differs between families and a multimodal stack nests one level deeper."""
    for path in ("model.language_model.layers", "model.model.language_model.layers", "model.layers", "model.model.layers"):
        node = model
        for attr in path.split("."):
            node = getattr(node, attr, None)
            if node is None:
                break
        if node is not None:
            return node
    raise AttributeError("decoder layers not found")


layers = decoder_layers(model)
LAYER = len(layers) // 2
print(f"слоёв {len(layers)}, берём {LAYER}")

## Сбор активаций

In [ ]:
def mean_activation(model, texts, layer):
    """Mean hidden state of `layer` over all positions and all texts, in float32."""
    captured = []
    handle = layers[layer].register_forward_hook(
        lambda module, args, output: captured.append((output[0] if isinstance(output, tuple) else output).float().mean(dim=1).squeeze(0).cpu()))
    try:
        with torch.no_grad():
            for text in texts:
                model(**tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device))
    finally:
        handle.remove()
    return torch.stack(captured).mean(dim=0)


def rendered(rows, key):
    return [tokenizer.apply_chat_template(r["prompt"] + r[key], tokenize=False, enable_thinking=False) for r in rows]


source = dev[:32]
mu_plus = mean_activation(model, rendered(source, "chosen"), LAYER)
mu_minus = mean_activation(model, rendered(source, "rejected"), LAYER)
vector = mu_plus - mu_minus
print(f"норма разности средних {vector.norm():.2f}, размерность {vector.numel()}")

## Хук и развёртка по силе

In [ ]:
@contextmanager
def steered(model, vector, alpha):
    """Add alpha * vector to the hidden states of LAYER for the duration of the block."""
    shift = (alpha * vector).to(model.device)

    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        shifted = hidden + shift.to(hidden.dtype)   # the model runs in bf16, the vector is kept in fp32
        return (shifted, *output[1:]) if isinstance(output, tuple) else shifted

    handle = layers[LAYER].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


row = next(r for r in extended if r["category"] == "thesis-intro-blocks")
print("ЗАПРОС:", data.request(row))
for alpha in (-1.0, 0.0, 1.0, 2.0):
    with steered(model, vector, alpha):
        answer = generate(model, [row], max_new_tokens=300)[0]
    print("═" * 78, f"α = {alpha:+.1f}")
    print(answer)

## Замер

Судью здесь не зовём: хук стоит и во время его работы и исказил бы вердикты.
Вектор — проверка гипотезы, а не кандидат в продакшен, ему хватает автопроверок и текста.

In [ ]:
for alpha in (1.0, 1.5):
    with steered(model, vector, alpha):
        result_e, _ = evaluate(model, extended, f"steer{alpha:.1f}-extended", note=f"steering, alpha {alpha}", with_judge=False)
    print(f"α = {alpha}:", report.table({f"α={alpha}": result_e}).splitlines()[-1])

torch.save({"vector": vector, "layer": LAYER}, "../runs/steering.pt")

Вектор двигает одно свойство и ничего не знает про остальные требования. Зато он строится
за минуту, снимается одной строкой и не портит веса, поэтому им удобно проверять, есть ли
в активациях нужное направление, до того как запускать дообучение.